In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader


# ------------------------------------------------------------
# 1. TOKEN DATA
# ------------------------------------------------------------

train_token_ids = torch.tensor(
    [0, 1, 2, 3, 1, 2, 4, 3, 0, 1, 2, 5],
    dtype=torch.long
)

validation_token_ids = torch.tensor(
    [0, 1, 2, 3, 1, 2, 5, 3],
    dtype=torch.long
)


# ------------------------------------------------------------
# 2. DATASET
# ------------------------------------------------------------

class NextTokenDataset(Dataset):

    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.token_ids) - self.block_size

    def __getitem__(self, index):
        x = self.token_ids[index:index + self.block_size]
        y = self.token_ids[index + 1:index + self.block_size + 1]

        return x, y


block_size = 4

trainset = NextTokenDataset(
    train_token_ids,
    block_size
)

validationset = NextTokenDataset(
    validation_token_ids,
    block_size
)


# ------------------------------------------------------------
# 3. DATALOADERS
# ------------------------------------------------------------

batch_size = 2

trainloader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

validationloader = DataLoader(
    validationset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)


# ------------------------------------------------------------
# 4. MODEL
# ------------------------------------------------------------

class TinyLanguageModel(nn.Module):

    def __init__(self, vocab_size, embedding_size):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )

        self.lm_head = nn.Linear(
            embedding_size,
            vocab_size
        )

    def forward(self, input_ids):
        embeddings = self.token_embedding(input_ids)
        logits = self.lm_head(embeddings)

        return logits


vocab_size = 6
embedding_size = 8

model = TinyLanguageModel(
    vocab_size,
    embedding_size
)


# ------------------------------------------------------------
# 5. DEVICE
# ------------------------------------------------------------

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model = model.to(device)


# ------------------------------------------------------------
# 6. LOSS AND OPTIMIZER
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001
)


# ------------------------------------------------------------
# 7. TRAINING
# ------------------------------------------------------------

num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for batch_index, (inputs, targets) in enumerate(trainloader):

        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        logits = model(inputs)

        B, T, V = logits.shape

        logits_flat = logits.reshape(B * T, V)
        targets_flat = targets.reshape(B * T)

        loss = criterion(
            logits_flat,
            targets_flat
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    average_training_loss = running_loss / len(trainloader)

    print(
        f"Epoch {epoch + 1}: "
        f"training loss = {average_training_loss:.4f}"
    )


# ------------------------------------------------------------
# 8. SAVE MODEL WEIGHTS
# ------------------------------------------------------------

PATH = "./tiny_language_model.pt"

torch.save(
    model.state_dict(),
    PATH
)


# ------------------------------------------------------------
# 9. VALIDATION
# ------------------------------------------------------------

model.eval()

validation_loss = 0.0

with torch.no_grad():

    for inputs, targets in validationloader:

        inputs = inputs.to(device)
        targets = targets.to(device)

        logits = model(inputs)

        B, T, V = logits.shape

        logits_flat = logits.reshape(B * T, V)
        targets_flat = targets.reshape(B * T)

        loss = criterion(
            logits_flat,
            targets_flat
        )

        validation_loss += loss.item()


average_validation_loss = (
    validation_loss / len(validationloader)
)

print(
    f"validation loss = "
    f"{average_validation_loss:.4f}"
)

Epoch 1: training loss = 1.6678
Epoch 2: training loss = 1.6511
Epoch 3: training loss = 1.6360
Epoch 4: training loss = 1.6198
Epoch 5: training loss = 1.6054
validation loss = 1.7413
